# DMET-SSTKG — Source-Side Motif Memory and Dirichlet-Conformal Hub Construction

This notebook reproduces the source-memory and calibrated evidence-hub
stage described in the manuscript section **Source-Side Motif Memory and
Calibrated Evidence Hub Construction**.

The notebook reuses the source S-STKG cache, the exact case-disjoint
split, and the retained phase-initialized motif-extractor checkpoint.
Stable source-training motifs are compressed into one normalized
prototype per phase. Source-validation motifs fit the Dirichlet
calibration map and the split-conformal Adaptive Prediction
Sets (APS) threshold. Source-test motifs
are used only for the post-construction audit.

No source-test or remote label is used to construct the phase prototypes
or estimate the calibration parameters.

## 1. Generated Artifacts

The hub-construction pipeline produces the following artifacts:

| Artifact | Description |
|---|---|
| `Shareable_Motif_Hub_DIRICHLET_CONFORMAL_PROTOTYPE.pt` | Serialized calibrated hub containing the normalized phase prototypes, Dirichlet calibration parameters, and split-conformal APS configuration. |
| `SourceHub_ConstructionSummary.csv` | Aggregate counts describing motif extraction, stable-anchor retention, phase-prototype construction, and serialized hub size. |
| `SourceHub_PhaseEvidenceSummary.csv` | Phase-level stable-anchor support, retention rates, and prototype compactness statistics. |
| `SourceHub_CalibrationAudit.csv` | Calibration and held-out audit metrics for the Dirichlet-conformal prototype classifier. |

In [ ]:
HUB_FILENAME = (
    "Shareable_Motif_Hub_"
    "DIRICHLET_CONFORMAL_PROTOTYPE.pt"
)

CONSTRUCTION_SUMMARY_FILENAME = (
    "SourceHub_ConstructionSummary.csv"
)
PHASE_EVIDENCE_FILENAME = (
    "SourceHub_PhaseEvidenceSummary.csv"
)
CALIBRATION_AUDIT_FILENAME = (
    "SourceHub_CalibrationAudit.csv"
)

## 2. Required Environment

```bash
pip install torch==2.4.1 torch-geometric pandas numpy scikit-learn tqdm
```

Select the PyTorch build appropriate for the local CPU or CUDA
environment.

In [ ]:
from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm
from torch_geometric.data import Data, Dataset
from torch_geometric.nn import GATv2Conv
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

## 3. Repository Paths

The paths are repository-relative examples and contain no user-specific
directory, video name, or case name. Replace `selected_motif_run` with
the output directory produced by the phase-initialized motif-extractor
notebook.

In [ ]:
# Replace the example paths below with the corresponding local directories.

from pathlib import Path

# Existing source role-normalized S-STKG directory.
# This folder must contain the source Case_* directories and their
# nodes, edges, edge_features, and labels files.
SSTKG_ROOT = Path(
    "/path/to/source_sstkg_directory"
)

# Directory used for the binary cache required by the HUB pipeline.
# No prebuilt cache is required. If this directory does not exist,
# the HUB code creates the cache from SSTKG_ROOT when it is run.
CACHE_DIR = Path(
    "/path/to/generated_hub_cache_directory"
)

# Completed motif-extractor run directory.
# Select the output directory produced by the phase-initialized
# motif-extractor code. It must contain best_motif_model.pt
# and data_split.npz.
MOTIF_RUN_DIR = Path(
    "/path/to/completed_motif_extractor_run"
)

# Motif-extractor checkpoint stored in the selected run directory.
MOTIF_CHECKPOINT = (
    MOTIF_RUN_DIR
    / "best_motif_model.pt"
)

# Source train/validation/test split copied into the motif run.
SPLIT_FILE = (
    MOTIF_RUN_DIR
    / "data_split.npz"
)

# Writable directory in which the calibrated HUB artifact and
# HUB-audit outputs will be saved. It may be a new directory.
HUB_OUTPUT_DIR = Path(
    "/path/to/hub_output_directory"
)

## 4. Hub and Calibration Configuration

The hub contains eight normalized 128-dimensional phase prototypes, a
multiclass Dirichlet calibration map, and a split-conformal APS threshold estimated from held-out source-validation motifs.

The conformal miscoverage level is fixed at `alpha=0.10`, corresponding to
nominal `1 - alpha = 0.90` coverage. This operating point follows commonly
used conformal-classification and adaptive prediction-set formulations [1, 2].
It is specified independently of source-test and remote outcomes and is not
selected by comparing downstream performance.

The value `alpha=0.10` defines the nominal miscoverage level; it is not the
deployed conformal threshold. The deployed threshold `qhat` is estimated from
the APS calibration scores of the held-out source-validation motifs. Each
calibration score represents the cumulative Dirichlet-calibrated probability
mass required to include the verified phase in the probability-ranked phase
list.

The source-training split provides the stable motifs used to construct the
phase prototypes. The source-validation split is used to fit the Dirichlet
calibration map and estimate `qhat`, whereas the source-test split is reserved
exclusively for post-construction audit.

In [ ]:
SEED = 42
DEVICE = torch.device("cpu")

CHUNK_SIZE = 100
MIN_CHUNK_FRAMES = 10

NUM_PHASES = 8
NODE_FEATURE_DIM = 11
EDGE_FEATURE_DIM = 3
HIDDEN_DIM = 128
MOTIF_DIM = 128

MOTIF_WINDOW = 16
MOTIF_STRIDE = 8

DIRICHLET_C = 1.0
DIRICHLET_MAX_ITER = 5000
CONFORMAL_ALPHA = 0.10

EPS = 1e-12
FORCE_REPROCESS_CACHE = False

PHASE_NAMES = [
    "Trocar Placement",
    "Preparation",
    "Calot Triangle Dissection",
    "Clipping and Cutting",
    "Gallbladder Dissection",
    "Gallbladder Packaging",
    "Cleaning and Coagulation",
    "Gallbladder Retraction",
]

PHASE_CODES = [
    f"P{phase_index}"
    for phase_index in range(1, NUM_PHASES + 1)
]

## 5. Node-Feature Interface

In [ ]:
# S-STKG node layout:
# x, y, w, h, vx, vy, class, case_id, g_ratio, p_ratio, isV
#
# Physical node classes are read directly from the constructed S-STKG.
# No physical-node class mapping is performed in this notebook.
#
# The bookkeeping case identifier is zeroed before model input, matching the
# source phase-classifier and motif-extractor stages.

REMOVE_CASE_ID_FEATURE = True
REMOVE_G_RATIO_FEATURE = False
REMOVE_P_RATIO_FEATURE = False

CASE_ID_COL = 7
G_RATIO_COL = 8
P_RATIO_COL = 9
VIRTUAL_FLAG_COL = 10

## 6. Reproducibility and Numerical Helpers

In [ ]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def phase_label_to_zero_based(
    raw_label,
    num_classes=8,
):
    raw_label = int(raw_label)

    if 1 <= raw_label <= num_classes:
        return raw_label - 1

    if 0 <= raw_label < num_classes:
        return raw_label

    raise ValueError(
        f"Unexpected phase label value: {raw_label}"
    )


def l2_normalize(
    values,
):
    values = np.asarray(
        values,
        dtype=np.float32,
    )

    if values.size == 0:
        return values

    if values.ndim == 1:
        return (
            values
            / (
                np.linalg.norm(values)
                + EPS
            )
        )

    return (
        values
        / (
            np.linalg.norm(
                values,
                axis=1,
                keepdims=True,
            )
            + EPS
        )
    )


def softmax_array(
    values,
    axis=-1,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )
    values = (
        values
        - np.max(
            values,
            axis=axis,
            keepdims=True,
        )
    )
    exponentials = np.exp(values)

    return (
        exponentials
        / (
            exponentials.sum(
                axis=axis,
                keepdims=True,
            )
            + EPS
        )
    )


def top2_margin(
    probabilities,
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )
    order = np.argsort(
        -probabilities
    )

    if len(order) == 1:
        return float(
            probabilities[
                order[0]
            ]
        )

    return float(
        probabilities[
            order[0]
        ]
        - probabilities[
            order[1]
        ]
    )


def window_spans(
    sequence_length,
    motif_window,
    motif_stride,
):
    sequence_length = int(
        sequence_length
    )

    if sequence_length < motif_window:
        return [
            (
                0,
                sequence_length - 1,
            )
        ]

    return [
        (
            start,
            start
            + motif_window
            - 1,
        )
        for start in range(
            0,
            sequence_length
            - motif_window
            + 1,
            motif_stride,
        )
    ]

## 7. Reusing the Source S-STKG Cache

The phase classifier and motif extractor convert the text-based S-STKG
files into a deterministic binary cache. The hub stage reuses that same
cache because the saved `data_split.npz` assigns fragments by cache
index. The cache is rebuilt only when the source S-STKG files or the
input-feature configuration have changed.

In [ ]:
class SSTKGSequenceDataset(Dataset):
    def __init__(
        self,
        root_dir,
        cache_dir,
        chunk_size=100,
        minimum_chunk_frames=10,
        force_reprocess=False,
    ):
        super().__init__(str(root_dir))

        self.root_dir = Path(root_dir)
        self.cache_dir = Path(cache_dir)
        self.chunk_size = int(chunk_size)
        self.minimum_chunk_frames = int(
            minimum_chunk_frames
        )

        if (
            force_reprocess
            and self.cache_dir.exists()
        ):
            shutil.rmtree(
                self.cache_dir
            )

        if (
            not self.cache_dir.exists()
            or not any(
                self.cache_dir.glob("*.pt")
            )
        ):
            self.cache_dir.mkdir(
                parents=True,
                exist_ok=True,
            )
            self._convert_and_fragment()
        else:
            print(
                "[OK] Existing source S-STKG "
                f"cache is reused: {self.cache_dir}"
            )

        self.files = sorted(
            self.cache_dir.glob("*.pt")
        )

        if not self.files:
            raise RuntimeError(
                f"Binary cache is empty: "
                f"{self.cache_dir}"
            )

    @staticmethod
    def _required_paths(
        case_directory,
        fragment_prefix,
    ):
        case_directory = Path(
            case_directory
        )

        return {
            "nodes": (
                case_directory
                / f"{fragment_prefix}_nodes.txt"
            ),
            "edges": (
                case_directory
                / f"{fragment_prefix}_edges.txt"
            ),
            "edge_features": (
                case_directory
                / (
                    f"{fragment_prefix}"
                    "_edge_features.txt"
                )
            ),
            "labels": (
                case_directory
                / f"{fragment_prefix}_labels.txt"
            ),
        }

    @staticmethod
    def _frame_mask(
        table,
        frame_progress,
    ):
        return np.isclose(
            table.iloc[:, 0].to_numpy(
                dtype=float
            ),
            float(frame_progress),
            rtol=0.0,
            atol=1e-6,
        )

    def _convert_and_fragment(self):
        if not self.root_dir.is_dir():
            raise FileNotFoundError(
                "S-STKG root directory was not "
                f"found: {self.root_dir}"
            )

        case_directories = sorted(
            path
            for path in self.root_dir.iterdir()
            if (
                path.is_dir()
                and path.name.startswith(
                    "Case_"
                )
            )
        )

        if not case_directories:
            raise RuntimeError(
                "No Case_XX directories were "
                f"found under {self.root_dir}"
            )

        fragment_index = 0

        for case_directory in tqdm(
            case_directories,
            desc="Building binary cache",
        ):
            fragment_prefixes = sorted(
                {
                    path.name.replace(
                        "_nodes.txt",
                        "",
                    )
                    for path in case_directory.glob(
                        "*_nodes.txt"
                    )
                }
            )

            for fragment_prefix in (
                fragment_prefixes
            ):
                paths = self._required_paths(
                    case_directory,
                    fragment_prefix,
                )

                missing_paths = [
                    path
                    for path in paths.values()
                    if not path.exists()
                ]

                if missing_paths:
                    print(
                        "[WARN] Incomplete fragment "
                        f"skipped: {fragment_prefix}"
                    )
                    continue

                node_table = pd.read_csv(
                    paths["nodes"]
                )
                edge_table = pd.read_csv(
                    paths["edges"]
                )
                edge_feature_table = pd.read_csv(
                    paths["edge_features"]
                )
                label_table = pd.read_csv(
                    paths["labels"]
                )

                for table in [
                    node_table,
                    edge_table,
                    edge_feature_table,
                    label_table,
                ]:
                    table.columns = [
                        str(column).strip()
                        for column in table.columns
                    ]

                expected_node_columns = [
                    "x",
                    "y",
                    "w",
                    "h",
                    "vx",
                    "vy",
                    "class",
                    "case_id",
                    "g_ratio",
                    "p_ratio",
                    "isV",
                ]

                if (
                    list(node_table.columns)
                    != expected_node_columns
                ):
                    raise ValueError(
                        "Unexpected node layout in "
                        f"{paths['nodes']}"
                    )

                if label_table.empty:
                    raise ValueError(
                        f"Empty label file: "
                        f"{paths['labels']}"
                    )

                raw_phase_label = int(
                    label_table.iloc[0, 1]
                )
                phase_label = (
                    phase_label_to_zero_based(
                        raw_phase_label,
                        NUM_PHASES,
                    )
                )

                progress_values = sorted(
                    node_table[
                        "g_ratio"
                    ].astype(float).unique()
                )

                for chunk_start in range(
                    0,
                    len(progress_values),
                    self.chunk_size,
                ):
                    chunk_values = (
                        progress_values[
                            chunk_start:
                            chunk_start
                            + self.chunk_size
                        ]
                    )

                    if (
                        len(chunk_values)
                        < self.minimum_chunk_frames
                    ):
                        continue

                    graph_sequence = []

                    for frame_progress in (
                        chunk_values
                    ):
                        node_mask = np.isclose(
                            node_table[
                                "g_ratio"
                            ].to_numpy(
                                dtype=float
                            ),
                            float(frame_progress),
                            rtol=0.0,
                            atol=1e-6,
                        )

                        frame_nodes = (
                            node_table.loc[
                                node_mask
                            ]
                            .to_numpy(
                                dtype=np.float32,
                                copy=True,
                            )
                        )

                        if REMOVE_CASE_ID_FEATURE:
                            frame_nodes[
                                :,
                                CASE_ID_COL,
                            ] = 0.0

                        if REMOVE_G_RATIO_FEATURE:
                            frame_nodes[
                                :,
                                G_RATIO_COL,
                            ] = 0.0

                        if REMOVE_P_RATIO_FEATURE:
                            frame_nodes[
                                :,
                                P_RATIO_COL,
                            ] = 0.0

                        edge_mask = (
                            self._frame_mask(
                                edge_table,
                                frame_progress,
                            )
                        )
                        edge_feature_mask = (
                            self._frame_mask(
                                edge_feature_table,
                                frame_progress,
                            )
                        )

                        frame_edges = (
                            edge_table.loc[
                                edge_mask
                            ].to_numpy()
                        )
                        frame_edge_features = (
                            edge_feature_table.loc[
                                edge_feature_mask
                            ].to_numpy()
                        )

                        if (
                            len(frame_edges)
                            != len(
                                frame_edge_features
                            )
                        ):
                            raise RuntimeError(
                                "Edge and edge-feature "
                                "rows are misaligned."
                            )

                        if len(frame_edges) > 0:
                            edge_index = torch.tensor(
                                frame_edges[
                                    :,
                                    2:4,
                                ].astype(
                                    np.int64
                                ),
                                dtype=torch.long,
                            ).t().contiguous()
                        else:
                            edge_index = torch.zeros(
                                (2, 0),
                                dtype=torch.long,
                            )

                        if (
                            len(
                                frame_edge_features
                            )
                            > 0
                        ):
                            edge_attr = torch.tensor(
                                frame_edge_features[
                                    :,
                                    4:7,
                                ].astype(
                                    np.float32
                                ),
                                dtype=torch.float,
                            )
                        else:
                            edge_attr = torch.zeros(
                                (
                                    0,
                                    EDGE_FEATURE_DIM,
                                ),
                                dtype=torch.float,
                            )

                        graph = Data(
                            x=torch.tensor(
                                frame_nodes,
                                dtype=torch.float,
                            ),
                            edge_index=edge_index,
                            edge_attr=edge_attr,
                            y=torch.tensor(
                                [phase_label],
                                dtype=torch.long,
                            ),
                        )
                        graph.is_v = (
                            graph.x[
                                :,
                                VIRTUAL_FLAG_COL,
                            ]
                            == 1
                        )

                        if int(
                            graph.is_v.sum().item()
                        ) != 1:
                            raise RuntimeError(
                                "Expected exactly one "
                                "virtual node per frame."
                            )

                        graph_sequence.append(
                            graph
                        )

                    torch.save(
                        {
                            "graph_seq": (
                                graph_sequence
                            ),
                            "metadata": {
                                "case_id": (
                                    case_directory.name
                                ),
                                "prefix": (
                                    fragment_prefix
                                ),
                                "phase_label_id": int(
                                    phase_label
                                ),
                            },
                        },
                        self.cache_dir
                        / (
                            "fragment_"
                            f"{fragment_index:06d}"
                            ".pt"
                        ),
                    )
                    fragment_index += 1

        print(
            "[OK] Binary fragments created: "
            f"{fragment_index}"
        )

    def len(self):
        return len(self.files)

    def get_with_metadata(
        self,
        index,
    ):
        cache_object = torch.load(
            self.files[int(index)],
            weights_only=False,
        )

        if (
            isinstance(cache_object, dict)
            and "graph_seq"
            in cache_object
        ):
            metadata = dict(
                cache_object.get(
                    "metadata",
                    {},
                )
                or {}
            )
            metadata[
                "fragment_file"
            ] = self.files[
                int(index)
            ].name

            return (
                cache_object[
                    "graph_seq"
                ],
                metadata,
            )

        return (
            cache_object,
            {
                "fragment_file": (
                    self.files[
                        int(index)
                    ].name
                ),
                "case_id": "UNKNOWN",
            },
        )

    def get(self, index):
        graph_sequence, _ = (
            self.get_with_metadata(
                index
            )
        )
        return graph_sequence

## 8. Reusing the Exact Source Split

The hub notebook does not create a fallback percentage split. It
requires the exact source train/validation/test assignment copied by the
motif-extractor stage and verifies that every cached fragment appears
once and only once.

In [ ]:
def load_saved_source_split(
    split_path,
    dataset_length,
):
    split_path = Path(split_path)

    if not split_path.exists():
        raise FileNotFoundError(
            f"Saved motif split was not "
            f"found: {split_path}"
        )

    split = np.load(split_path)

    required_keys = {
        "train_idx",
        "val_idx",
        "test_idx",
    }

    if not required_keys.issubset(
        split.files
    ):
        raise ValueError(
            "The split file must contain "
            "train_idx, val_idx, and test_idx."
        )

    train_indices = (
        split["train_idx"]
        .astype(int)
    )
    validation_indices = (
        split["val_idx"]
        .astype(int)
    )
    test_indices = (
        split["test_idx"]
        .astype(int)
    )

    all_indices = np.concatenate(
        [
            train_indices,
            validation_indices,
            test_indices,
        ]
    )

    if len(all_indices) != dataset_length:
        raise ValueError(
            "The saved split does not contain "
            "exactly one assignment for every "
            "cached source fragment."
        )

    if (
        all_indices.min() < 0
        or all_indices.max()
        >= dataset_length
    ):
        raise ValueError(
            "The saved split is incompatible "
            "with the current cache length."
        )

    if len(
        np.unique(all_indices)
    ) != dataset_length:
        raise ValueError(
            "The saved split contains overlap "
            "or duplicate fragment indices."
        )

    split_lookup = {
        int(index): "train"
        for index in train_indices
    }
    split_lookup.update(
        {
            int(index): "validation"
            for index in validation_indices
        }
    )
    split_lookup.update(
        {
            int(index): "test"
            for index in test_indices
        }
    )

    return (
        train_indices,
        validation_indices,
        test_indices,
        split_lookup,
    )

## 9. Phase-Initialized Motif Extractor

The architecture matches the preceding motif-extractor notebook. A small
checkpoint-key compatibility function accepts either the original
development naming convention or the GitHub notebook naming convention.
This compatibility step concerns model parameter names only; it does not
alter S-STKG node classes.

In [ ]:
class PhaseInitializedSSTKGMotifExtractor(
    nn.Module
):
    def __init__(
        self,
        in_channels=11,
        edge_dim=3,
        hidden_dim=128,
        motif_dim=128,
        num_phases=8,
        motif_window=16,
        motif_stride=8,
    ):
        super().__init__()

        self.hidden_dim = int(
            hidden_dim
        )
        self.motif_window = int(
            motif_window
        )
        self.motif_stride = int(
            motif_stride
        )

        self.graph_conv_1 = GATv2Conv(
            in_channels,
            hidden_dim,
            edge_dim=edge_dim,
        )
        self.graph_conv_2 = GATv2Conv(
            hidden_dim,
            hidden_dim,
            edge_dim=edge_dim,
        )
        self.virtual_context_gate = nn.Linear(
            hidden_dim * 2,
            1,
        )
        self.temporal_encoder = nn.GRU(
            hidden_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.phase_head = nn.Linear(
            hidden_dim * 3,
            num_phases,
        )

        self.window_fusion = nn.Linear(
            hidden_dim * 2,
            hidden_dim,
        )
        self.motif_projection = nn.Sequential(
            nn.Linear(
                hidden_dim,
                hidden_dim,
            ),
            nn.ReLU(),
            nn.Linear(
                hidden_dim,
                motif_dim,
            ),
        )
        self.motif_phase_head = nn.Linear(
            motif_dim,
            num_phases,
        )

    def encode_frame_graphs(
        self,
        graph_sequence,
    ):
        frame_embeddings = []

        for graph in graph_sequence:
            node_embeddings = (
                self.graph_conv_1(
                    graph.x,
                    graph.edge_index,
                    graph.edge_attr,
                )
                .relu()
            )
            node_embeddings = (
                self.graph_conv_2(
                    node_embeddings,
                    graph.edge_index,
                    graph.edge_attr,
                )
                .relu()
            )

            mean_embedding = (
                node_embeddings.mean(
                    dim=0,
                    keepdim=True,
                )
            )
            virtual_embedding = (
                node_embeddings[
                    graph.is_v
                ]
            )

            if virtual_embedding.size(0) == 0:
                virtual_embedding = (
                    mean_embedding
                )
            else:
                virtual_embedding = (
                    virtual_embedding.mean(
                        dim=0,
                        keepdim=True,
                    )
                )

            gate_weight = torch.sigmoid(
                self.virtual_context_gate(
                    torch.cat(
                        [
                            mean_embedding,
                            virtual_embedding,
                        ],
                        dim=-1,
                    )
                )
            )

            frame_embeddings.append(
                gate_weight
                * virtual_embedding
                + (
                    1.0
                    - gate_weight
                )
                * mean_embedding
            )

        return (
            torch.stack(
                frame_embeddings
            )
            .squeeze(1)
        )

    def contextualize_frames(
        self,
        frame_embeddings,
    ):
        recurrent_output, hidden_state = (
            self.temporal_encoder(
                frame_embeddings.unsqueeze(0)
            )
        )
        recurrent_output = (
            recurrent_output.squeeze(0)
        )

        forward_output = (
            recurrent_output[
                :,
                :self.hidden_dim,
            ]
        )
        backward_output = (
            recurrent_output[
                :,
                self.hidden_dim:,
            ]
        )

        temporal_context = (
            0.5
            * (
                forward_output
                + backward_output
            )
        )
        contextualized_frames = (
            0.5
            * frame_embeddings
            + 0.5
            * temporal_context
        )

        final_recurrent_state = torch.cat(
            [
                hidden_state[0],
                hidden_state[1],
            ],
            dim=-1,
        )
        temporal_mean_skip = (
            frame_embeddings.mean(
                dim=0,
                keepdim=True,
            )
        )
        sequence_readout = torch.cat(
            [
                final_recurrent_state,
                temporal_mean_skip,
            ],
            dim=-1,
        )

        return (
            contextualized_frames,
            sequence_readout,
        )

    def fuse_window(
        self,
        window_embeddings,
    ):
        return self.window_fusion(
            torch.cat(
                [
                    window_embeddings.mean(
                        dim=0,
                        keepdim=True,
                    ),
                    window_embeddings[
                        -1
                    ].unsqueeze(0),
                ],
                dim=-1,
            )
        )

    def extract_motif_signatures(
        self,
        contextualized_frames,
    ):
        sequence_length = int(
            contextualized_frames.shape[0]
        )

        if sequence_length < self.motif_window:
            fused_windows = self.fuse_window(
                contextualized_frames
            )
        else:
            fused_windows = torch.cat(
                [
                    self.fuse_window(
                        contextualized_frames[
                            start:
                            start
                            + self.motif_window
                        ]
                    )
                    for start in range(
                        0,
                        sequence_length
                        - self.motif_window
                        + 1,
                        self.motif_stride,
                    )
                ],
                dim=0,
            )

        return F.normalize(
            self.motif_projection(
                fused_windows
            ),
            p=2,
            dim=-1,
        )

    def forward(
        self,
        graph_sequence,
    ):
        frame_embeddings = (
            self.encode_frame_graphs(
                graph_sequence
            )
        )
        (
            contextualized_frames,
            sequence_readout,
        ) = self.contextualize_frames(
            frame_embeddings
        )
        motif_signatures = (
            self.extract_motif_signatures(
                contextualized_frames
            )
        )

        return {
            "motif_logits": (
                self.motif_phase_head(
                    motif_signatures
                )
            ),
            "motif_signatures": (
                motif_signatures
            ),
            "sequence_logits": (
                self.phase_head(
                    sequence_readout
                )
            ),
        }

In [ ]:
def _convert_development_checkpoint_keys(
    state_dict,
):
    prefix_map = {
        "conv1.": "graph_conv_1.",
        "conv2.": "graph_conv_2.",
        "context_attn.": (
            "virtual_context_gate."
        ),
        "rnn.": "temporal_encoder.",
        "fc.": "phase_head.",
        "window_fuse.": "window_fusion.",
        "motif_proj.": "motif_projection.",
        "motif_cls.": "motif_phase_head.",
    }

    converted = {}

    for key, value in state_dict.items():
        converted_key = key

        for old_prefix, new_prefix in (
            prefix_map.items()
        ):
            if key.startswith(old_prefix):
                converted_key = (
                    new_prefix
                    + key[
                        len(old_prefix):
                    ]
                )
                break

        converted[
            converted_key
        ] = value

    return converted


def load_motif_model(
    checkpoint_path,
):
    checkpoint_path = Path(
        checkpoint_path
    )

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            "Motif-extractor checkpoint "
            f"was not found: {checkpoint_path}"
        )

    model = (
        PhaseInitializedSSTKGMotifExtractor(
            in_channels=NODE_FEATURE_DIM,
            edge_dim=EDGE_FEATURE_DIM,
            hidden_dim=HIDDEN_DIM,
            motif_dim=MOTIF_DIM,
            num_phases=NUM_PHASES,
            motif_window=MOTIF_WINDOW,
            motif_stride=MOTIF_STRIDE,
        )
        .to(DEVICE)
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    state_dict = checkpoint.get(
        "model_state_dict",
        checkpoint,
    )
    state_dict = (
        _convert_development_checkpoint_keys(
            state_dict
        )
    )

    model.load_state_dict(
        state_dict,
        strict=True,
    )
    model.eval()

    return model

## 10. Source Motif Extraction

Every source fragment produces sequence-level and motif-level phase
predictions together with normalized motif signatures. The verified
parent-fragment phase is retained locally to determine stable anchors
and to perform source-side calibration and audit.

In [ ]:
def extract_source_motif_records(
    model,
    dataset,
    split_lookup,
):
    fragment_rows = []
    motif_rows = []
    motif_embeddings = []

    with torch.no_grad():
        for dataset_index in tqdm(
            range(len(dataset)),
            desc="Extracting source motifs",
        ):
            (
                graph_sequence_cpu,
                metadata,
            ) = dataset.get_with_metadata(
                dataset_index
            )
            graph_sequence = [
                graph.to(DEVICE)
                for graph in graph_sequence_cpu
            ]

            model_output = model(
                graph_sequence
            )

            true_phase_id = int(
                graph_sequence_cpu[0]
                .y.item()
            )

            sequence_probabilities = (
                softmax_array(
                    model_output[
                        "sequence_logits"
                    ]
                    .detach()
                    .cpu()
                    .numpy()[0]
                )
            )
            sequence_prediction = int(
                np.argmax(
                    sequence_probabilities
                )
            )

            motif_probabilities = (
                softmax_array(
                    model_output[
                        "motif_logits"
                    ]
                    .detach()
                    .cpu()
                    .numpy(),
                    axis=1,
                )
            )
            motif_predictions = (
                motif_probabilities.argmax(
                    axis=1
                )
            )
            motif_signatures = (
                l2_normalize(
                    model_output[
                        "motif_signatures"
                    ]
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                )
            )

            spans = window_spans(
                len(graph_sequence_cpu),
                model.motif_window,
                model.motif_stride,
            )

            if len(spans) != len(
                motif_signatures
            ):
                raise RuntimeError(
                    "Motif-window and signature "
                    "counts do not match."
                )

            split_name = split_lookup[
                int(dataset_index)
            ]
            fragment_id = len(
                fragment_rows
            )

            fragment_rows.append(
                {
                    "fragment_id": int(
                        fragment_id
                    ),
                    "dataset_index": int(
                        dataset_index
                    ),
                    "split": split_name,
                    "case_id": str(
                        metadata.get(
                            "case_id",
                            "UNKNOWN",
                        )
                    ),
                    "prefix": str(
                        metadata.get(
                            "prefix",
                            "",
                        )
                    ),
                    "true_phase_id": int(
                        true_phase_id
                    ),
                    "sequence_prediction": int(
                        sequence_prediction
                    ),
                    "num_motifs": int(
                        len(
                            motif_signatures
                        )
                    ),
                }
            )

            for (
                local_motif_index,
                (
                    span,
                    signature,
                    motif_prediction,
                ),
            ) in enumerate(
                zip(
                    spans,
                    motif_signatures,
                    motif_predictions,
                )
            ):
                motif_id = len(
                    motif_rows
                )
                motif_embeddings.append(
                    signature
                )

                motif_rows.append(
                    {
                        "motif_id": int(
                            motif_id
                        ),
                        "fragment_id": int(
                            fragment_id
                        ),
                        "dataset_index": int(
                            dataset_index
                        ),
                        "split": (
                            split_name
                        ),
                        "case_id": str(
                            metadata.get(
                                "case_id",
                                "UNKNOWN",
                            )
                        ),
                        "prefix": str(
                            metadata.get(
                                "prefix",
                                "",
                            )
                        ),
                        "local_motif_index": int(
                            local_motif_index
                        ),
                        "window_start": int(
                            span[0]
                        ),
                        "window_end": int(
                            span[1]
                        ),
                        "true_phase_id": int(
                            true_phase_id
                        ),
                        "sequence_prediction": int(
                            sequence_prediction
                        ),
                        "motif_prediction": int(
                            motif_prediction
                        ),
                    }
                )

    return (
        pd.DataFrame(
            fragment_rows
        ),
        pd.DataFrame(
            motif_rows
        ),
        l2_normalize(
            np.asarray(
                motif_embeddings,
                dtype=np.float32,
            )
        ),
    )

## 11. Stable Anchors and Phase Prototypes

A source-training motif is retained only when the verified fragment
phase, the sequence-head prediction, and the motif-head prediction
agree. Stable motifs remain local. For each phase, their normalized mean
forms one normalized source prototype.

In [ ]:
def add_stability_flag(
    motif_table,
):
    motif_table = motif_table.copy()

    motif_table[
        "is_stable_anchor"
    ] = (
        (
            motif_table[
                "sequence_prediction"
            ].astype(int)
            == motif_table[
                "true_phase_id"
            ].astype(int)
        )
        & (
            motif_table[
                "motif_prediction"
            ].astype(int)
            == motif_table[
                "true_phase_id"
            ].astype(int)
        )
        & (
            motif_table[
                "sequence_prediction"
            ].astype(int)
            == motif_table[
                "motif_prediction"
            ].astype(int)
        )
    ).astype(int)

    return motif_table


def build_phase_prototypes(
    motif_table,
    motif_embeddings,
):
    motif_table = (
        add_stability_flag(
            motif_table
        )
    )

    training_motifs = motif_table[
        motif_table[
            "split"
        ]
        == "train"
    ].copy()

    stable_training_motifs = (
        training_motifs[
            training_motifs[
                "is_stable_anchor"
            ]
            == 1
        ]
        .copy()
    )

    if stable_training_motifs.empty:
        raise RuntimeError(
            "No stable source-training "
            "motifs were found."
        )

    prototypes = []
    records = []
    phase_rows = []

    for phase_id in range(
        NUM_PHASES
    ):
        phase_training = (
            training_motifs[
                training_motifs[
                    "true_phase_id"
                ].astype(int)
                == phase_id
            ]
        )
        phase_stable = (
            stable_training_motifs[
                stable_training_motifs[
                    "true_phase_id"
                ].astype(int)
                == phase_id
            ]
        )

        if phase_stable.empty:
            raise RuntimeError(
                "No stable training motif "
                f"was found for "
                f"{PHASE_CODES[phase_id]} "
                f"{PHASE_NAMES[phase_id]}."
            )

        motif_indices = (
            phase_stable[
                "motif_id"
            ].to_numpy(
                dtype=int
            )
        )
        phase_embeddings = (
            l2_normalize(
                motif_embeddings[
                    motif_indices
                ]
            )
        )
        prototype = l2_normalize(
            phase_embeddings.mean(
                axis=0
            )
        )
        similarities = (
            phase_embeddings
            @ prototype.reshape(
                -1,
                1,
            )
        ).reshape(-1)

        prototypes.append(
            prototype.astype(
                np.float32
            )
        )

        records.append(
            {
                "prototype_id": int(
                    phase_id
                ),
                "verified_phase_id": int(
                    phase_id
                ),
                "verified_phase_code": (
                    PHASE_CODES[
                        phase_id
                    ]
                ),
                "verified_phase_name": (
                    PHASE_NAMES[
                        phase_id
                    ]
                ),
                "num_stable_train_anchors": int(
                    len(
                        phase_stable
                    )
                ),
                "mean_within_phase_similarity": float(
                    similarities.mean()
                ),
                "std_within_phase_similarity": float(
                    similarities.std()
                ),
            }
        )

        retention = (
            len(phase_stable)
            / max(
                len(phase_training),
                1,
            )
        )

        phase_rows.append(
            {
                "phase_id": int(
                    phase_id
                ),
                "phase_code": (
                    PHASE_CODES[
                        phase_id
                    ]
                ),
                "phase_name": (
                    PHASE_NAMES[
                        phase_id
                    ]
                ),
                "total_train_motifs": int(
                    len(
                        phase_training
                    )
                ),
                "stable_train_anchors": int(
                    len(
                        phase_stable
                    )
                ),
                "anchor_retention_percent": float(
                    100.0
                    * retention
                ),
                "prototype_cosine_mean": float(
                    similarities.mean()
                ),
                "prototype_cosine_sd": float(
                    similarities.std()
                ),
            }
        )

    prototype_embeddings = (
        l2_normalize(
            np.asarray(
                prototypes,
                dtype=np.float32,
            )
        )
    )

    return (
        pd.DataFrame(records),
        prototype_embeddings,
        motif_table,
        stable_training_motifs,
        pd.DataFrame(
            phase_rows
        ),
    )

## 12. Dirichlet Calibration and Split-Conformal APS

Prototype cosine similarities are converted to raw phase probabilities.
A multiclass Dirichlet calibration map is fitted on source-validation
motifs in log-probability space. The calibrated validation probabilities
then determine the APS threshold at `alpha=0.10`.

The final audit table reports only the measures used in the paper:
number of motifs, top-1 accuracy, NLL, ECE, conformal coverage, and mean
candidate-set size.

In [ ]:
def prototype_probabilities(
    query_embeddings,
    prototype_embeddings,
):
    similarities = (
        l2_normalize(
            query_embeddings
        )
        @ l2_normalize(
            prototype_embeddings
        ).T
    )

    return softmax_array(
        similarities,
        axis=1,
    )


def fit_dirichlet_calibrator(
    calibration_embeddings,
    calibration_labels,
    prototype_embeddings,
):
    calibration_labels = np.asarray(
        calibration_labels,
        dtype=int,
    )

    if len(calibration_labels) == 0:
        raise RuntimeError(
            "No source-validation motifs "
            "were available for calibration."
        )

    if len(
        np.unique(
            calibration_labels
        )
    ) != NUM_PHASES:
        raise RuntimeError(
            "Dirichlet calibration requires "
            "all source phases in the "
            "validation split."
        )

    raw_probabilities = np.clip(
        prototype_probabilities(
            calibration_embeddings,
            prototype_embeddings,
        ),
        EPS,
        1.0,
    )
    log_probabilities = np.log(
        raw_probabilities
    )

    calibrator = LogisticRegression(
        solver="lbfgs",
        C=DIRICHLET_C,
        max_iter=DIRICHLET_MAX_ITER,
        class_weight="balanced",
        random_state=SEED,
    )
    calibrator.fit(
        log_probabilities,
        calibration_labels,
    )

    return {
        "type": (
            "multiclass_dirichlet_"
            "calibration"
        ),
        "classes": (
            calibrator.classes_
            .astype(int)
            .tolist()
        ),
        "coef": (
            calibrator.coef_
            .astype(float)
            .tolist()
        ),
        "intercept": (
            calibrator.intercept_
            .astype(float)
            .tolist()
        ),
        "C": float(
            DIRICHLET_C
        ),
        "max_iter": int(
            DIRICHLET_MAX_ITER
        ),
    }


def apply_dirichlet_calibrator(
    query_embeddings,
    prototype_embeddings,
    calibrator,
):
    raw_probabilities = np.clip(
        prototype_probabilities(
            query_embeddings,
            prototype_embeddings,
        ),
        EPS,
        1.0,
    )
    log_probabilities = np.log(
        raw_probabilities
    )

    coefficients = np.asarray(
        calibrator["coef"],
        dtype=np.float64,
    )
    intercept = np.asarray(
        calibrator["intercept"],
        dtype=np.float64,
    )
    classes = np.asarray(
        calibrator["classes"],
        dtype=int,
    )

    calibrated_logits = (
        log_probabilities
        @ coefficients.T
        + intercept.reshape(
            1,
            -1,
        )
    )
    observed_probabilities = (
        softmax_array(
            calibrated_logits,
            axis=1,
        )
    )

    probabilities = np.zeros(
        (
            len(
                query_embeddings
            ),
            NUM_PHASES,
        ),
        dtype=np.float64,
    )

    for column_index, phase_id in enumerate(
        classes
    ):
        probabilities[
            :,
            int(phase_id),
        ] = observed_probabilities[
            :,
            column_index,
        ]

    return (
        probabilities
        / (
            probabilities.sum(
                axis=1,
                keepdims=True,
            )
            + EPS
        )
    )


def aps_scores(
    probabilities,
    labels,
):
    scores = []

    for probability, label in zip(
        np.asarray(
            probabilities,
            dtype=np.float64,
        ),
        np.asarray(
            labels,
            dtype=int,
        ),
    ):
        order = np.argsort(
            -probability
        )
        cumulative_probability = (
            np.cumsum(
                probability[
                    order
                ]
            )
        )
        true_position = int(
            np.where(
                order
                == int(label)
            )[0][0]
        )
        scores.append(
            float(
                cumulative_probability[
                    true_position
                ]
            )
        )

    return np.asarray(
        scores,
        dtype=np.float64,
    )


def fit_aps_threshold(
    calibrated_probabilities,
    calibration_labels,
    alpha=0.10,
):
    scores = aps_scores(
        calibrated_probabilities,
        calibration_labels,
    )
    number_of_scores = len(
        scores
    )

    if number_of_scores == 0:
        raise RuntimeError(
            "No APS calibration scores "
            "were available."
        )

    quantile_level = min(
        1.0,
        np.ceil(
            (
                number_of_scores
                + 1
            )
            * (
                1.0
                - float(alpha)
            )
        )
        / number_of_scores,
    )

    threshold = float(
        np.quantile(
            scores,
            quantile_level,
            method="higher",
        )
    )

    return {
        "type": (
            "split_conformal_APS"
        ),
        "alpha": float(
            alpha
        ),
        "target_coverage": float(
            1.0
            - alpha
        ),
        "qhat": threshold,
        "num_calibration_records": int(
            number_of_scores
        ),
    }


def conformal_sets(
    probabilities,
    threshold,
):
    candidate_sets = []
    candidate_sizes = []

    for probability in np.asarray(
        probabilities,
        dtype=np.float64,
    ):
        order = np.argsort(
            -probability
        )
        cumulative_probability = (
            np.cumsum(
                probability[
                    order
                ]
            )
        )

        crossing_positions = np.where(
            cumulative_probability
            > float(threshold)
        )[0]

        if len(crossing_positions) == 0:
            final_position = (
                len(order)
                - 1
            )
        else:
            final_position = int(
                crossing_positions[0]
            )

        selected = (
            order[
                :final_position + 1
            ]
            .astype(int)
            .tolist()
        )
        selected = sorted(
            set(selected)
        )

        candidate_sets.append(
            selected
        )
        candidate_sizes.append(
            len(selected)
        )

    return (
        candidate_sets,
        np.asarray(
            candidate_sizes,
            dtype=int,
        ),
    )


def expected_calibration_error(
    labels,
    probabilities,
    number_of_bins=10,
):
    labels = np.asarray(
        labels,
        dtype=int,
    )
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    confidence = probabilities.max(
        axis=1
    )
    prediction = probabilities.argmax(
        axis=1
    )
    correct = (
        prediction
        == labels
    ).astype(float)

    bin_edges = np.linspace(
        0.0,
        1.0,
        number_of_bins + 1,
    )
    calibration_error = 0.0

    for bin_index in range(
        number_of_bins
    ):
        lower = bin_edges[
            bin_index
        ]
        upper = bin_edges[
            bin_index + 1
        ]

        if (
            bin_index
            == number_of_bins - 1
        ):
            mask = (
                (confidence >= lower)
                & (confidence <= upper)
            )
        else:
            mask = (
                (confidence >= lower)
                & (confidence < upper)
            )

        if mask.sum() == 0:
            continue

        calibration_error += (
            float(mask.mean())
            * abs(
                float(
                    correct[
                        mask
                    ].mean()
                )
                - float(
                    confidence[
                        mask
                    ].mean()
                )
            )
        )

    return float(
        calibration_error
    )


def evaluate_calibrated_hub(
    split_name,
    query_embeddings,
    labels,
    prototype_embeddings,
    dirichlet_calibrator,
    conformal_calibrator,
):
    labels = np.asarray(
        labels,
        dtype=int,
    )
    probabilities = (
        apply_dirichlet_calibrator(
            query_embeddings,
            prototype_embeddings,
            dirichlet_calibrator,
        )
    )
    predictions = probabilities.argmax(
        axis=1
    )
    candidate_sets, candidate_sizes = (
        conformal_sets(
            probabilities,
            conformal_calibrator[
                "qhat"
            ],
        )
    )

    coverage = float(
        np.mean(
            [
                int(label)
                in candidate_set
                for (
                    label,
                    candidate_set,
                )
                in zip(
                    labels,
                    candidate_sets,
                )
            ]
        )
    )

    negative_log_likelihood = (
        float(
            log_loss(
                labels,
                probabilities,
                labels=list(
                    range(
                        NUM_PHASES
                    )
                ),
            )
        )
    )

    return {
        "split": split_name,
        "num_motifs": int(
            len(labels)
        ),
        "top1_accuracy": float(
            np.mean(
                predictions
                == labels
            )
        ),
        "nll": (
            negative_log_likelihood
        ),
        "ece": (
            expected_calibration_error(
                labels,
                probabilities,
            )
        ),
        "conformal_coverage": (
            coverage
        ),
        "mean_conformal_set_size": float(
            candidate_sizes.mean()
        ),
    }


def split_motif_arrays(
    motif_table,
    motif_embeddings,
    split_name,
):
    split_rows = (
        motif_table[
            motif_table[
                "split"
            ]
            == split_name
        ]
        .copy()
    )

    motif_indices = (
        split_rows[
            "motif_id"
        ].to_numpy(
            dtype=int
        )
    )

    return (
        l2_normalize(
            motif_embeddings[
                motif_indices
            ]
        ),
        split_rows[
            "true_phase_id"
        ].to_numpy(
            dtype=int
        ),
    )

## 13. Serialized Dirichlet-Conformal Hub Artifact

The calibrated hub is serialized as a compact artifact containing eight
L2-normalized phase prototypes, their fixed P1–P8 identifiers, the fitted
Dirichlet calibration parameters, and the split-conformal APS threshold `qhat`.

Individual source motifs, case identifiers, full S-STKG sequences, model
parameters, and calibration samples are not included.

In [ ]:
def build_shareable_hub_object(
    prototype_records,
    prototype_embeddings,
    dirichlet_calibrator,
    conformal_calibrator,
    calibration_audit,
):
    phase_ids = list(
        range(
            NUM_PHASES
        )
    )

    hub_object = {
        "bank_type": (
            "DMET_SSTKG_"
            "DirichletConformalPrototypeHub"
        ),
        "version": (
            "source_phase_prototype_hub_v1"
        ),
        "phase_ids": phase_ids,
        "phase_names": PHASE_NAMES,
        "num_records": int(
            len(
                prototype_records
            )
        ),
        "embedding_dim": int(
            prototype_embeddings.shape[1]
        ),
        "records": (
            prototype_records.to_dict(
                orient="records"
            )
        ),
        "embeddings": (
            l2_normalize(
                prototype_embeddings
            )
            .astype(np.float32)
        ),
        "calibration": {
            "temperature_scaling_used": (
                False
            ),
            "dirichlet_calibrator": (
                dirichlet_calibrator
            ),
            "conformal_calibrator": (
                conformal_calibrator
            ),
            "metrics": (
                calibration_audit.to_dict(
                    orient="records"
                )
            ),
        },
        "construction": {
            "prototype_source_split": (
                "train"
            ),
            "calibration_split": (
                "validation"
            ),
            "source_test_role": (
                "post_construction_audit_only"
            ),
            "stable_anchor_rule": (
                "verified_phase_equals_"
                "sequence_prediction_equals_"
                "motif_prediction"
            ),
            "prototype_rule": (
                "L2_normalized_mean_of_"
                "stable_training_motifs"
            ),
            "conformal_alpha": float(
                CONFORMAL_ALPHA
            ),
            "case_id_input_zeroed": bool(
                REMOVE_CASE_ID_FEATURE
            ),
        },
        "privacy_statement": {
            "contains_raw_video": False,
            "contains_raw_frames": False,
            "contains_full_sstkg_sequences": (
                False
            ),
            "contains_case_identifiers": False,
            "contains_model_parameters": False,
            "contains_individual_source_motifs": (
                False
            ),
            "contains_phase_prototypes": True,
            "contains_calibration_parameters": (
                True
            ),
        },
    }

    if hub_object[
        "phase_ids"
    ] != phase_ids:
        raise RuntimeError(
            "Prototype phase ordering is "
            "not P1-P8."
        )

    if (
        hub_object[
            "embeddings"
        ].shape
        != (
            NUM_PHASES,
            MOTIF_DIM,
        )
    ):
        raise RuntimeError(
            "Unexpected hub prototype shape: "
            f"{hub_object['embeddings'].shape}"
        )

    return hub_object


def build_construction_summary(
    motif_table,
    stable_training_motifs,
    prototype_embeddings,
    hub_path,
):
    hub_bytes = int(
        Path(hub_path).stat().st_size
    )

    return pd.DataFrame(
        [
            {
                "source_motif_records": int(
                    len(
                        motif_table
                    )
                ),
                "stable_train_anchors": int(
                    len(
                        stable_training_motifs
                    )
                ),
                "phase_prototypes": int(
                    len(
                        prototype_embeddings
                    )
                ),
                "serialized_hub_bytes": int(
                    hub_bytes
                ),
                "serialized_hub_mib": float(
                    hub_bytes
                    / (
                        1024.0
                        ** 2
                    )
                ),
            }
        ]
    )


def save_required_outputs(
    hub_object,
    construction_summary,
    phase_evidence_summary,
    calibration_audit,
):
    HUB_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    expected_names = {
        HUB_FILENAME,
        CONSTRUCTION_SUMMARY_FILENAME,
        PHASE_EVIDENCE_FILENAME,
        CALIBRATION_AUDIT_FILENAME,
    }

    for existing_file in (
        HUB_OUTPUT_DIR.iterdir()
    ):
        if (
            existing_file.is_file()
            and existing_file.name
            in expected_names
        ):
            existing_file.unlink()

    hub_path = (
        HUB_OUTPUT_DIR
        / HUB_FILENAME
    )
    torch.save(
        hub_object,
        hub_path,
    )

    construction_summary = (
        build_construction_summary(
            motif_table=construction_summary[
                "motif_table"
            ],
            stable_training_motifs=(
                construction_summary[
                    "stable_training_motifs"
                ]
            ),
            prototype_embeddings=(
                hub_object[
                    "embeddings"
                ]
            ),
            hub_path=hub_path,
        )
    )

    construction_summary.to_csv(
        HUB_OUTPUT_DIR
        / CONSTRUCTION_SUMMARY_FILENAME,
        index=False,
    )
    phase_evidence_summary.to_csv(
        HUB_OUTPUT_DIR
        / PHASE_EVIDENCE_FILENAME,
        index=False,
    )
    calibration_audit.to_csv(
        HUB_OUTPUT_DIR
        / CALIBRATION_AUDIT_FILENAME,
        index=False,
    )

    generated_files = sorted(
        path.name
        for path in HUB_OUTPUT_DIR.iterdir()
        if (
            path.is_file()
            and path.name
            in expected_names
        )
    )

    if set(
        generated_files
    ) != expected_names:
        raise RuntimeError(
            "The expected output contract "
            "was not satisfied."
        )

    return {
        "hub": hub_path,
        "construction_summary": (
            HUB_OUTPUT_DIR
            / CONSTRUCTION_SUMMARY_FILENAME
        ),
        "phase_evidence_summary": (
            HUB_OUTPUT_DIR
            / PHASE_EVIDENCE_FILENAME
        ),
        "calibration_audit": (
            HUB_OUTPUT_DIR
            / CALIBRATION_AUDIT_FILENAME
        ),
    }

## 14. Run Hub Construction

The final cell saves only the four files declared in the output
contract. It also reloads the serialized hub and verifies the fields
required by remote consultation.

In [ ]:
def run_source_hub_pipeline():
    set_seed(SEED)

    required_paths = [
        MOTIF_CHECKPOINT,
        SPLIT_FILE,
    ]

    missing_paths = [
        path
        for path in required_paths
        if not Path(path).exists()
    ]

    if missing_paths:
        raise FileNotFoundError(
            "Required motif-extractor "
            f"artifacts are missing: "
            f"{missing_paths}"
        )

    print(
        "\n"
        + "=" * 88
    )
    print(
        "SOURCE-SIDE MOTIF MEMORY AND "
        "DIRICHLET-CONFORMAL HUB"
    )
    print(
        "=" * 88
    )
    print(
        f"S-STKG root: {SSTKG_ROOT}"
    )
    print(
        f"Shared cache: {CACHE_DIR}"
    )
    print(
        f"Motif checkpoint: "
        f"{MOTIF_CHECKPOINT}"
    )
    print(
        f"Saved split: {SPLIT_FILE}"
    )
    print(
        f"Output directory: "
        f"{HUB_OUTPUT_DIR}"
    )

    dataset = SSTKGSequenceDataset(
        root_dir=SSTKG_ROOT,
        cache_dir=CACHE_DIR,
        chunk_size=CHUNK_SIZE,
        minimum_chunk_frames=(
            MIN_CHUNK_FRAMES
        ),
        force_reprocess=(
            FORCE_REPROCESS_CACHE
        ),
    )

    (
        train_indices,
        validation_indices,
        test_indices,
        split_lookup,
    ) = load_saved_source_split(
        SPLIT_FILE,
        len(dataset),
    )

    model = load_motif_model(
        MOTIF_CHECKPOINT
    )

    (
        fragment_table,
        motif_table,
        motif_embeddings,
    ) = extract_source_motif_records(
        model,
        dataset,
        split_lookup,
    )

    (
        prototype_records,
        prototype_embeddings,
        motif_table,
        stable_training_motifs,
        phase_evidence_summary,
    ) = build_phase_prototypes(
        motif_table,
        motif_embeddings,
    )

    (
        validation_embeddings,
        validation_labels,
    ) = split_motif_arrays(
        motif_table,
        motif_embeddings,
        "validation",
    )

    dirichlet_calibrator = (
        fit_dirichlet_calibrator(
            validation_embeddings,
            validation_labels,
            prototype_embeddings,
        )
    )
    validation_probabilities = (
        apply_dirichlet_calibrator(
            validation_embeddings,
            prototype_embeddings,
            dirichlet_calibrator,
        )
    )
    conformal_calibrator = (
        fit_aps_threshold(
            validation_probabilities,
            validation_labels,
            alpha=CONFORMAL_ALPHA,
        )
    )

    (
        test_embeddings,
        test_labels,
    ) = split_motif_arrays(
        motif_table,
        motif_embeddings,
        "test",
    )

    calibration_audit = pd.DataFrame(
        [
            evaluate_calibrated_hub(
                split_name=(
                    "source_validation_calibration"
                ),
                query_embeddings=(
                    validation_embeddings
                ),
                labels=(
                    validation_labels
                ),
                prototype_embeddings=(
                    prototype_embeddings
                ),
                dirichlet_calibrator=(
                    dirichlet_calibrator
                ),
                conformal_calibrator=(
                    conformal_calibrator
                ),
            ),
            evaluate_calibrated_hub(
                split_name=(
                    "held_out_source_test_audit"
                ),
                query_embeddings=(
                    test_embeddings
                ),
                labels=(
                    test_labels
                ),
                prototype_embeddings=(
                    prototype_embeddings
                ),
                dirichlet_calibrator=(
                    dirichlet_calibrator
                ),
                conformal_calibrator=(
                    conformal_calibrator
                ),
            ),
        ]
    )

    hub_object = (
        build_shareable_hub_object(
            prototype_records=(
                prototype_records
            ),
            prototype_embeddings=(
                prototype_embeddings
            ),
            dirichlet_calibrator=(
                dirichlet_calibrator
            ),
            conformal_calibrator=(
                conformal_calibrator
            ),
            calibration_audit=(
                calibration_audit
            ),
        )
    )

    output_paths = save_required_outputs(
        hub_object=hub_object,
        construction_summary={
            "motif_table": (
                motif_table
            ),
            "stable_training_motifs": (
                stable_training_motifs
            ),
        },
        phase_evidence_summary=(
            phase_evidence_summary
        ),
        calibration_audit=(
            calibration_audit
        ),
    )

    saved_hub = torch.load(
        output_paths["hub"],
        map_location="cpu",
        weights_only=False,
    )

    required_remote_keys = [
        "embeddings",
        "records",
        "calibration",
    ]

    missing_remote_keys = [
        key
        for key in required_remote_keys
        if key not in saved_hub
    ]

    if missing_remote_keys:
        raise RuntimeError(
            "The saved hub is missing "
            "remote-consultation fields: "
            f"{missing_remote_keys}"
        )

    calibration_fields = (
        saved_hub[
            "calibration"
        ]
    )

    if (
        "dirichlet_calibrator"
        not in calibration_fields
        or "conformal_calibrator"
        not in calibration_fields
    ):
        raise RuntimeError(
            "The saved hub does not contain "
            "the calibration objects required "
            "by remote consultation."
        )

    print(
        "\n[GENERATED FILES]"
    )
    print(
        "Remote consultation:"
    )
    print(
        f"  {output_paths['hub']}"
    )
    print(
        "Paper — hub construction funnel:"
    )
    print(
        f"  {output_paths['construction_summary']}"
    )
    print(
        "Paper — phase-wise evidence support:"
    )
    print(
        f"  {output_paths['phase_evidence_summary']}"
    )
    print(
        "Paper — validation calibration and "
        "held-out source-test audit:"
    )
    print(
        f"  {output_paths['calibration_audit']}"
    )

    return {
        "dataset": dataset,
        "model": model,
        "fragment_table": (
            fragment_table
        ),
        "motif_table": (
            motif_table
        ),
        "stable_training_motifs": (
            stable_training_motifs
        ),
        "phase_evidence_summary": (
            phase_evidence_summary
        ),
        "calibration_audit": (
            calibration_audit
        ),
        "hub_object": (
            saved_hub
        ),
        "output_paths": (
            output_paths
        ),
    }

In [ ]:
hub_run = run_source_hub_pipeline()

## 15. Using the Outputs

- The remote notebook or script loads only
  `Shareable_Motif_Hub_DIRICHLET_CONFORMAL_PROTOTYPE.pt`.
- `SourceHub_ConstructionSummary.csv` supplies the values for the paper's
  hub-construction funnel.
- `SourceHub_PhaseEvidenceSummary.csv` supplies phase-level stable-anchor
  and prototype-compactness values.
- `SourceHub_CalibrationAudit.csv` supplies the validation-calibration
  and held-out source-test audit values.

No other runtime output is required by the manuscript or the remote
consultation pipeline.

## References

[1] Romano, Y., Sesia, M., & Candès, E. J. (2020). Classification with
valid and adaptive coverage. *Advances in Neural Information Processing
Systems, 33*, 3581–3591.

[2] Angelopoulos, A. N., Bates, S., Jordan, M. I., & Malik, J. (2021).
Uncertainty sets for image classifiers using conformal prediction.
*International Conference on Learning Representations*.